# What temperature does a probe in a hot gas measure?

A small bead in a duct exchanges energy with both the gas and the duct walls. Calculate the indicated temperature from an energy balance, infer the gas temperature from the reading, and test how emissivity changes the measurement.

## Draw the control surface around the bead

Let the gas have temperature $T_g$ and the bead a uniform temperature $T_s$. A large, isothermal enclosure at $T_w$ fills the bead's field of view. Use kelvins throughout. Convection supplies the bead with heat at $h A(T_g-T_s)$. A diffuse grey bead emits $\epsilon\sigma A T_s^4$ and absorbs $\epsilon\sigma A T_w^4$ from the enclosure. The absorptivity and emissivity are equal for the grey surface. The common area $A$ cancels from the steady energy balance when axial conduction through the support is negligible.

$$F(T_s)=h(T_g-T_s)-\epsilon\sigma(T_s^4-T_w^4)=0.$$

For a hotter gas, $F(T_w)$ is positive, $F(T_g)$ is nonpositive, and $F'(T_s)=-h-4\epsilon\sigma T_s^3$ is strictly negative. There is exactly one equilibrium between the two reservoir temperatures. A colder gas reverses their ordering and keeps the same argument. This physical bracket gives a bisection solver two trustworthy endpoints. With positive heat capacity, a small rise in bead temperature makes the net heat input negative, which restores equilibrium.

The factorization $T_s^4-T_w^4=(T_s-T_w)(T_s+T_w)(T_s^2+T_w^2)$ gives an exact coefficient $h_r=\epsilon\sigma(T_s+T_w)(T_s^2+T_w^2)$. At the root, $T_g-T_s=h_r(T_s-T_w)/h$. A shiny surface and a fast gas stream therefore act on different terms of the same equation. A larger bead changes the transient response through its mass-to-area ratio, and may also change $h$. Area alone cannot correct the steady reading at fixed $h$.

## Predict the indicated temperature

Take $T_g=420$ K, $T_w=300$ K, $h=25$ W m$^{-2}$ K$^{-1}$ and $\epsilon=0.8$. At 390 K, convection supplies 750 W m$^{-2}$ and the radiative loss is approximately 682 W m$^{-2}$. The bead thus warms. At 400 K the convective input has fallen to 500 W m$^{-2}$ while radiative loss reaches 794 W m$^{-2}$. The intersection gives $T_s=391.90$ K, an under-reading of 28.10 K. The two independently evaluated fluxes at equilibrium agree to round-off. Doubling $h$ gives 403.34 K for the same gas and wall. Metal lead wires connected to a cold mount could supply a third heat path, which the two-flux residual cannot independently diagnose.

## Set the experimental inputs

Edit these values, then run the calculation and both figures again.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

teal, orange, blue = "#19596a", "#aa462a", "#3868a0"
def axes(xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.grid(alpha=0.2)
    return fig, ax

In [ ]:
gas_K = 420.0
wall_K = 300.0
film_W_m2K = 25.0
emissivity = 0.8
sigma = 5.670374419e-8

In [ ]:
assert min(gas_K, wall_K, film_W_m2K) > 0 and 0 <= emissivity <= 1
def indicated(gas, wall, film, eps):
    lower, upper = sorted((gas, wall))
    for _ in range(70):
        trial = (lower + upper) / 2
        net_input = film*(gas - trial) - eps*sigma*(trial**4 - wall**4)
        if net_input > 0:
            lower = trial
        else:
            upper = trial
    return (lower + upper)/2

sensor_K = indicated(gas_K, wall_K, film_W_m2K, emissivity)
convective = film_W_m2K*(gas_K-sensor_K)
radiative = emissivity*sigma*(sensor_K**4-wall_K**4)
assert np.isclose(convective, radiative, rtol=1e-9, atol=1e-7)
print(f"Indicated temperature: {sensor_K:.2f} K; gas minus reading: {gas_K-sensor_K:.2f} K")
print(f"Convection in: {convective:.2f} W/m²; radiation out: {radiative:.2f} W/m²")

In [ ]:
trial = np.linspace(max(1, min(wall_K, gas_K)-5), max(wall_K, gas_K)+5, 250)
fig, ax = axes("Trial bead temperature (K)", "Heat flux (W/m²)")
ax.plot(trial, film_W_m2K*(gas_K-trial), color=teal, label="Convection into bead")
ax.plot(trial, emissivity*sigma*(trial**4-wall_K**4), color=orange, label="Radiation leaving bead")
ax.scatter([sensor_K], [convective], color="black", zorder=3, label=f"Equilibrium {sensor_K:.1f} K")
ax.legend()
plt.show()

Follow the curves to the same bead temperature. Their vertical separation is the energy available to warm or cool the bead.

In [ ]:
films = np.geomspace(max(1e-3, film_W_m2K/8), film_W_m2K*8, 120)
fig, ax = axes("Convection coefficient (W/m² K)", "Gas minus indicated temperature (K)")
for eps, color in zip((0.1, 0.5, 0.9), (teal, blue, orange)):
    biases = [gas_K-indicated(gas_K, wall_K, h, eps) for h in films]
    ax.semilogx(films, biases, color=color, label=f"Emissivity {eps:.1f}")
ax.scatter([film_W_m2K], [gas_K-sensor_K], color="black", label="Selected probe")
ax.legend()
plt.show()

Track one surface along its curve to change the flow, or hold the heat-transfer coefficient fixed and compare surface finishes.

## Problem 1 · Correct a recorded temperature

The probe reads 400 K. The walls remain at 300 K, $h=25$ W m$^{-2}$ K$^{-1}$ and $\epsilon=0.8$. Recover the gas temperature without iteration. How would a hotter enclosure affect the sign of this correction?

Rearranging the balance gives $T_g=T_s+\epsilon\sigma(T_s^4-T_w^4)/h$. The correction is $793.85/25=31.75$ K, giving 431.75 K. If the walls were hotter than the bead, absorbed radiation would exceed emitted radiation and the correction to the reading would be negative. A Celsius temperature can be reported at the end after the fourth powers have been evaluated in kelvins.

## Problem 2 · Specify a minimum convection coefficient

For the original gas at 420 K, find $h$ required for the under-reading to be at most 5 K. Then reduce the emissivity to 0.2 and repeat. Explain the ratio.

At the limiting reading $T_s=415$ K the required coefficient is $\epsilon\sigma(415^4-300^4)/5$. For $\epsilon=0.8$ this is approximately 196 W m$^{-2}$ K$^{-1}$. The required coefficient scales directly with emissivity at this prescribed reading, so the lower-emissivity bead needs about 49 W m$^{-2}$ K$^{-1}$. Above those coefficients the gas transfers more heat at the limiting temperature and drives the equilibrium closer to 420 K.